# 14. SCENIC+ Preparation — Configurable Seed, Downsampling & Cell Type Subsetting

Prepare SCENIC+ input data and Snakemake run directories with:
- **Multiple seeds** for robustness testing
- **Downsampling** all species to identical cell type composition (match-to-min or fixed N)
- **Cell type subsetting** to run SCENIC+ on specific cell types only

Each combination of (species, seed, downsample) gets its own Snakemake output directory.

In [2]:
import os
import re
import yaml
import subprocess
import numpy as np
import pandas as pd
import scanpy as sc
from pathlib import Path

# Add src to path
PIPELINE_DIR = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(PIPELINE_DIR))

from src.scenicplus_prep import (
    load_prep_config,
    compute_downsample_targets,
    build_run_name,
    build_sub_rna_name,
    generate_config_yaml as build_config_yaml,
    init_snakemake_dir,
    write_config,
    write_run_manifest,
    generate_batch_script,
)

/local1/scratch/jjans/miniforge3/envs/scenicplus_scenicplus10a2_py311/lib/python3.11/site-packages/pycisTopic/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


## Configuration

Edit this cell to set up your run.

In [3]:
# ============================================================
# MAIN CONFIGURATION — edit config/scenicplus_prep.yaml

# Optional: set PREP_CONFIG_PATH to another YAML file before running this cell.
# ============================================================

if "PREP_CONFIG_PATH" not in globals():
    PREP_CONFIG_PATH = str(Path("../config/scenicplus_prep.yaml").resolve())

prep_cfg = load_prep_config(PREP_CONFIG_PATH)
PATHS = prep_cfg["paths"]
RUN_CFG = prep_cfg["run"]
SPECIES_CONFIG = prep_cfg["species_config"]
SHARED_TASKS = prep_cfg.get("shared_tasks", {})

BASE_DIR = PATHS["base_dir"]
SCENICPLUS_DIR = PATHS["scenicplus_dir"]
INPUT_FILES_DIR = PATHS["input_files_dir"]
METADATA_DIR = PATHS["metadata_dir"]
TEMP_DIR = PATHS["temp_dir"]
LOGS_DIR = PATHS["logs_dir"]
MANIFESTS_DIR = PATHS["manifests_dir"]
SCRIPTS_DIR = PATHS["scripts_dir"]

SPECIES_LIST = RUN_CFG["species_list"]
SEEDS = RUN_CFG["seeds"]
CELL_TYPES_SUBSET = RUN_CFG.get("cell_type_subset")
DOWNSAMPLE_MODE = RUN_CFG.get("downsample_mode", "min")
DOWNSAMPLE_N = RUN_CFG.get("downsample_n")
N_CPU = int(RUN_CFG["n_cpu"])
CELL_TYPE_COL = RUN_CFG["cell_type_col"]
AUTO_EXECUTE_BATCH = bool(RUN_CFG.get("auto_execute_batch", False))
SNAKEMAKE_EXTRA_ARGS = RUN_CFG.get("snakemake_extra_args", "--use-conda")

for d in [INPUT_FILES_DIR, TEMP_DIR, LOGS_DIR, MANIFESTS_DIR, SCRIPTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Config file:   {PREP_CONFIG_PATH}")
print(f"SCENIC+ dir:   {SCENICPLUS_DIR}")
print(f"Seeds:         {SEEDS}")
print(f"Cell types:    {CELL_TYPES_SUBSET or 'all'}")
print(f"Downsample:    {DOWNSAMPLE_MODE}")
print(f"Auto execute:  {AUTO_EXECUTE_BATCH}")

Config file:   /mp/nfs4krb5/bs-nas02/bsse_group_treutlein/USERS/jjans/analysis/adult_intestine/peaks/peak_calling/atac_pipeline/config/scenicplus_prep.yaml
SCENIC+ dir:   /home/jjanssens/jjans/analysis/adult_intestine/scenicplus
Seeds:         [42, 123, 666]
Cell types:    all
Downsample:    min
Auto execute:  False


## Species-specific configuration

Each species has different database paths, genome annotations, and biomart species codes. These are taken from the existing working configs.

In [4]:
# Species-specific settings are now loaded from config/scenicplus_prep.yaml
print(f"Loaded species configs: {len(SPECIES_CONFIG)}")
display(pd.DataFrame.from_dict(SPECIES_CONFIG, orient="index"))

Loaded species configs: 6


,biomart_species,ctx_db,dem_db,motif_annotations,region_set_folder,genome_annotation,chromsizes
Human,hsapiens,/home/jjanssens/jjans/resources/resources.aert...,/home/jjanssens/jjans/resources/resources.aert...,/home/jjanssens/jjans/resources/resources.aert...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
Gorilla,ggorilla,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/resources/resources.aert...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
Chimpanzee,ptroglodytes,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/resources/resources.aert...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
Bonobo,ppaniscus,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/resources/resources.aert...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
Macaque,mmulatta,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/resources/resources.aert...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
Marmoset,cjacchus,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/resources/resources.aert...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...


## Load cell type annotations and compute composition

In [5]:
import pandas as pd
import re

def reformat_index(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardizes indices from multiple formats:
    Format A: 'Sample_086_plus_resequenced_BARCODE-1' -> 'BARCODE-1-Sample_086'
    Format B: 'B006-A-002#BARCODE-1'                  -> 'BARCODE-1-B006_A_002'
    """
    s = df.index.to_series().astype(str)
    
    sample_id = pd.Series(index=s.index, dtype=str)
    barcode = pd.Series(index=s.index, dtype=str)
    
    # --- 1. Handle Format B (The '#' Delimiter) ---
    mask_hash = s.str.contains("#")
    if mask_hash.any():
        split_hash = s[mask_hash].str.split("#", n=1, expand=True)
        
        # FIX: Replace hyphens with underscores in the sample ID ONLY
        sample_id[mask_hash] = split_hash[0].str.replace("-", "_") 
        
        barcode[mask_hash] = split_hash[1]    # Keep the barcode (and its -1) intact

    # --- 2. Handle Format A (The '_' Delimiter & Legacy Prefixes) ---
    mask_underscore = ~mask_hash
    if mask_underscore.any():
        s_old = s[mask_underscore]
        
        # Normalize legacy prefixes
        s_old = s_old.str.replace(r"^(MSN_?|Sample_*)", "Sample_", regex=True)
        s_old = s_old.str.replace(r"^Sample(?=\d)", "Sample_", regex=True)
        
        # Extract components non-greedily
        extracted = s_old.str.extract(r"^(Sample_\d+).*?_([A-Z]+-\d+)$")
        
        if extracted.isna().any().any():
            bad_indices = s_old[extracted[0].isna()].head(5).tolist()
            raise ValueError(f"Could not parse these underscore barcodes: {bad_indices}")
            
        sample_id[mask_underscore] = extracted[0]
        barcode[mask_underscore] = extracted[1]

    # --- 3. Reconstruct ---
    df_out = df.copy()
    df_out.index = barcode + "-" + sample_id
    
    return df_out

In [6]:
import pandas as pd

def duplicate_sample_name_in_index(df: pd.DataFrame) -> pd.DataFrame:
    """
    Takes an index formatted as 'BARCODE-1-SampleID' and duplicates the sample ID.
    Converts: 'AAACAGCCAACTAGCC-1-B006-A-002'
    To:       'AAACAGCCAACTAGCC-1-B006-A-002___B006-A-002'
    """
    s = df.index.to_series().astype(str)
    
    # Safely extract the Sample ID.
    # We DO NOT split by standard '-' because 'B006-A-002' contains hyphens.
    # Instead, we split exactly at the '-1-' that connects the barcode to the sample ID.
    parts = s.str.split('-1-', n=1, expand=True)
    
    # parts[0] is the 16bp barcode (e.g., 'AAACAGCCAACTAGCC')
    # parts[1] is the safely extracted Sample ID (e.g., 'B006-A-002' or 'Sample_086')
    sample_ids = parts[1]
    
    # Catch any potential malformed strings where the split failed
    if sample_ids.isna().any():
        bad = s[sample_ids.isna()].head().tolist()
        raise ValueError(f"Could not find the '-1-' anchor to extract sample ID in: {bad}")
    
    # Append the triple underscore and the sample ID
    df_out = df.copy()
    df_out.index = s + "___" + sample_ids
    
    return df_out

In [7]:
# Load cell type annotations from RNA metadata
cell_meta = {}
for species in SPECIES_LIST:
    meta_path = os.path.join(METADATA_DIR, f"{species}_RNA.tsv")
    df = pd.read_csv(meta_path, sep="\t", index_col=0, usecols=lambda c: c in ["", CELL_TYPE_COL, "cell_type_initial"])
    # The first column (index) has no header name in some files — reload just what we need
    df_full = pd.read_csv(meta_path, sep="\t", index_col=0)
    df_full = duplicate_sample_name_in_index(reformat_index(df_full))
    cell_meta[species] = df_full[[CELL_TYPE_COL]].copy()
    print(f"{species}: {len(cell_meta[species]):,} cells, {cell_meta[species][CELL_TYPE_COL].nunique()} cell types")

# Also load the existing SCENIC+ input RNA to know which cells are actually available
available_cells = {}
for species in SPECIES_LIST:
    rna_path = os.path.join(INPUT_FILES_DIR, f"{species}_adata_rna_sub.h5ad")
    adata = sc.read_h5ad(rna_path, backed="r")
    available_cells[species] = set(adata.obs_names.tolist())
    print(f"{species}: {len(available_cells[species]):,} cells in SCENIC+ input")
    adata.file.close()

Human: 92,527 cells, 32 cell types


/tmp/ipykernel_170015/796240993.py:7: DtypeWarning: Columns (50) have mixed types. Specify dtype option on import or set low_memory=False.
  df_full = pd.read_csv(meta_path, sep="\t", index_col=0)


Gorilla: 51,911 cells, 33 cell types
Chimpanzee: 58,937 cells, 32 cell types
Bonobo: 57,814 cells, 35 cell types
Macaque: 62,330 cells, 32 cell types
Marmoset: 131,505 cells, 33 cell types
Human: 82,197 cells in SCENIC+ input
Gorilla: 29,569 cells in SCENIC+ input
Chimpanzee: 36,485 cells in SCENIC+ input
Bonobo: 28,756 cells in SCENIC+ input
Macaque: 37,497 cells in SCENIC+ input
Marmoset: 75,417 cells in SCENIC+ input


In [ ]:
df_full = cell_meta['Marmoset']

In [7]:
import pandas as pd

qc_records = []

for species in SPECIES_LIST:
    # 1. Get Barcodes
    meta_barcodes = pd.Series(cell_meta[species].index)
    adata_barcodes = pd.Series(list(available_cells[species]))
    
    # Intersection of exact barcode strings
    overlap_barcodes = pd.Series(list(set(meta_barcodes).intersection(available_cells[species])))
    
    # 2. Extract Sample IDs (Everything after the triple underscore)
    meta_samples = meta_barcodes.str.split('___').str[-1]
    adata_samples = adata_barcodes.str.split('___').str[-1]
    overlap_samples = overlap_barcodes.str.split('___').str[-1]
    
    # 3. Count cells per sample
    meta_counts = meta_samples.value_counts().rename('Metadata_Cells')
    adata_counts = adata_samples.value_counts().rename('AnnData_Cells')
    overlap_counts = overlap_samples.value_counts().rename('Overlap_Cells')
    
    # 4. Merge counts into a single DataFrame for this species
    df_species = pd.concat([meta_counts, adata_counts, overlap_counts], axis=1).fillna(0).astype(int)
    
    # Add species and sample ID as standard columns
    df_species.insert(0, 'Species', species)
    df_species = df_species.reset_index().rename(columns={'index': 'Sample_ID'})
    
    qc_records.append(df_species)

# Combine all species into one master QC DataFrame
qc_df = pd.concat(qc_records, ignore_index=True)

# Add a helpful percentage column to quickly spot drop-offs
qc_df['%_AnnData_in_Metadata'] = ((qc_df['Overlap_Cells'] / qc_df['AnnData_Cells']) * 100).round(2)
qc_df['%_Metadata_in_AnnData'] = ((qc_df['Overlap_Cells'] / qc_df['Metadata_Cells']) * 100).round(2)

# Display the result
print(qc_df.to_string())

     Sample_ID     Species  Metadata_Cells  AnnData_Cells  Overlap_Cells  %_AnnData_in_Metadata  %_Metadata_in_AnnData
0   B010_A_201       Human            8450           7837           7837                  100.0                  92.75
1   B010_A_501       Human            7002           6718           6718                  100.0                  95.94
2   B012_A_501       Human            5794           5219           5219                  100.0                  90.08
3   B012_A_101       Human            5576           5123           5123                  100.0                  91.88
4   B008_A_002       Human            5481           5011           5011                  100.0                  91.42
5   B012_A_001       Human            5309           4718           4718                  100.0                  88.87
6   B012_A_002       Human            4770           4258           4258                  100.0                  89.27
7   B010_A_002       Human            4488      

In [8]:
# Build cross-species cell type composition table
# We need to match cell metadata indices to the available cells in SCENIC+ input
composition = {}
for species in SPECIES_LIST:
    meta = cell_meta[species]
    avail = available_cells[species]
    # Find cells present in both metadata and SCENIC+ input
    common = meta.index.intersection(list(avail))
    ct_counts = meta.loc[common, CELL_TYPE_COL].value_counts()
    composition[species] = ct_counts

comp_df = pd.DataFrame(composition).fillna(0).astype(int)
comp_df = comp_df.sort_index()

print("Cell type composition across species (cells available in SCENIC+ input):")
display(comp_df)
print(f"\nTotal cells per species: {comp_df.sum().to_dict()}")

Cell type composition across species (cells available in SCENIC+ input):


,Human,Gorilla,Chimpanzee,Bonobo,Macaque,Marmoset
Adipocytes,39,6,26,141,31,45
BEST4+ cells,1436,33,22,48,439,177
Colonocytes,4809,422,463,977,1620,1407
Crypt Fibroblasts WNT2B+,2315,1170,2855,673,1314,4239
ECs,1223,519,752,513,278,2307
EECs,712,230,191,235,271,997
Enteric glia,1011,970,282,127,183,594
Enteric neurons,49,2,0,9,9,224
Enterocytes,4432,440,1613,1904,1062,18750
Eosinophils,0,432,0,46,0,0



Total cells per species: {'Human': 82197, 'Gorilla': 29569, 'Chimpanzee': 36485, 'Bonobo': 28756, 'Macaque': 37497, 'Marmoset': 75417}


In [9]:
# Apply cell type subset filter if specified
if CELL_TYPES_SUBSET is not None:
    missing = [ct for ct in CELL_TYPES_SUBSET if ct not in comp_df.index]
    if missing:
        print(f"WARNING: These cell types are not in the data: {missing}")
    comp_df = comp_df.loc[comp_df.index.intersection(CELL_TYPES_SUBSET)]
    print(f"Filtered to {len(comp_df)} cell types: {list(comp_df.index)}")

target_per_ct, ds_label = compute_downsample_targets(
    comp_df=comp_df,
    mode=DOWNSAMPLE_MODE,
    downsample_n=DOWNSAMPLE_N,

)

if target_per_ct is None:
    print("\nNo downsampling — using all available cells.")
elif ds_label == "dsMin":
    print("\nDownsampling target per cell type (match-to-min):")
    display(target_per_ct)
else:
    print(f"\nDownsampling target: {int(target_per_ct.iloc[0])} cells per type")


Downsampling target per cell type (match-to-min):


Adipocytes                                6
BEST4+ cells                             22
Colonocytes                             422
Crypt Fibroblasts WNT2B+                673
ECs                                     278
EECs                                    191
Enteric glia                            127
Enteric neurons                           2
Enterocytes                             440
Eosinophils                              46
Goblet cells                            681
Goblet-like cells                       220
ICCs                                      6
ILCs                                     83
Lymphatic ECs                           174
MARCO+ Lymphatic ECs                     44
MUC6+ cells                             221
Macrophages                             862
Mast cells                               56
Mesothelial cells                       109
Monocytes                                76
Myofibroblasts                          133
Naive B cells                   

## Downsampling function

In [10]:
def downsample_cells(meta_df, cell_type_col, target_per_ct, seed, available_barcodes=None):
    """
    Downsample cells to target counts per cell type.
    
    Parameters
    ----------
    meta_df : DataFrame with cell_type_col column
    cell_type_col : column name for cell type
    target_per_ct : Series mapping cell_type -> target N, or None for no downsampling
    seed : random seed
    available_barcodes : optional set of barcodes to restrict to
    
    Returns
    -------
    List of selected cell barcodes
    """
    rng = np.random.RandomState(seed)
    
    if available_barcodes is not None:
        meta_df = meta_df.loc[meta_df.index.intersection(list(available_barcodes))]
    
    if target_per_ct is None:
        return meta_df.index.tolist()
    
    selected = []
    for ct in target_per_ct.index:
        ct_cells = meta_df[meta_df[cell_type_col] == ct].index.tolist()
        n_target = min(target_per_ct[ct], len(ct_cells))
        if n_target > 0 and len(ct_cells) > 0:
            chosen = rng.choice(ct_cells, size=n_target, replace=False)
            selected.extend(chosen)
    
    return selected

In [13]:
import numpy as np
import pandas as pd

def downsample_cells(meta_df, cell_type_col, target_per_ct, seed, available_barcodes=None, min_cells=50):
    """
    Downsample cells to target counts per cell type, completely excluding 
    cell types that fall below the minimum threshold.
    
    Parameters
    ----------
    meta_df : DataFrame with cell_type_col column
    cell_type_col : column name for cell type
    target_per_ct : Series mapping cell_type -> target N, or None for no upper limit downsampling
    seed : random seed
    available_barcodes : optional set of barcodes to restrict to
    min_cells : int, default 50. Minimum number of available cells required to keep the cell type.
    
    Returns
    -------
    List of selected cell barcodes
    """
    rng = np.random.RandomState(seed)
    
    # 1. Restrict to available barcodes if provided
    if available_barcodes is not None:
        meta_df = meta_df.loc[meta_df.index.intersection(list(available_barcodes))]
    
    # 2. If no target downsampling is requested, just filter out the rare cell types
    if target_per_ct is None:
        if min_cells > 0:
            counts = meta_df[cell_type_col].value_counts()
            valid_cts = counts[counts >= min_cells].index
            return meta_df[meta_df[cell_type_col].isin(valid_cts)].index.tolist()
        else:
            return meta_df.index.tolist()
    
    # 3. Target downsampling with minimum threshold enforcement
    selected = []
    for ct in target_per_ct.index:
        ct_cells = meta_df[meta_df[cell_type_col] == ct].index.tolist()
        
        # Check against the minimum cell threshold
        if len(ct_cells) < min_cells:
            # Skip this cell type entirely (it will not be in the final output)
            continue
            
        n_target = min(target_per_ct[ct], len(ct_cells))
        if n_target > 0:
            chosen = rng.choice(ct_cells, size=n_target, replace=False)
            selected.extend(chosen)
            
    return selected

## Barcode transformation helpers

Replicates the species-specific barcode handling from the existing 00_/10_ notebooks.

In [11]:
def duplicate_sample_name_in_index(df, inplace=False):
    """
    Human barcode transform: <BARCODE>-Sample_XXX -> <BARCODE>-Sample_XXX___Sample_XXX
    Used by the SCENIC+ pipeline for Human data.
    """
    def modify_index(entry):
        match = re.search(r"(.*-B.*)", entry)
        if match:
            sample = match.group(1).split("-")[-1]
            return f"{entry}___{sample}"
        return entry
    
    if inplace:
        df.index = df.index.map(modify_index)
        return None
    else:
        new_df = df.copy()
        new_df.index = new_df.index.map(modify_index)
        return new_df


def reformat_index_nhp(df):
    """
    Non-human primate barcode transform for SCENIC+ pipeline.
    Handles Sample_086_BARCODE -> BARCODE-Sample_086 style naming.
    """
    s = df.index.to_series()
    s = s.str.replace(r"^MSN_?", "Sample_", regex=True)
    s = s.str.replace(r"^Sample(?=\d)", "Sample_", regex=True)
    s = s.str.replace(r"^Sample__+", "Sample_", regex=True)
    s = s.str.replace(r"_plus_resequenced_", "_", regex=True)
    s = s.str.replace(r"_plus_resequenced$", "", regex=True)
    
    m = s.str.extract(r"^(Sample_\d+)_(.+)$")
    if m.isna().any().any():
        bad = s[m[0].isna()].tolist()
        raise ValueError(f"Unexpected index format for: {bad[:5]}")
    
    new_index = m[1] + "-" + m[0]
    out = df.copy()
    out.index = new_index
    return out

## Config YAML generation

In [12]:
def generate_config_yaml(species, seed, cistopic_path, rna_path, species_cfg, temp_dir, n_cpu):
    """Compatibility wrapper around src.scenicplus_prep.generate_config_yaml."""
    return build_config_yaml(
        species=species,
        seed=seed,
        cistopic_path=cistopic_path,
        rna_path=rna_path,
        species_cfg=species_cfg,
        temp_dir=temp_dir,
        n_cpu=n_cpu,
    )

## Generate runs

For each species × seed combination:
1. Load the existing RNA AnnData
2. Apply cell type filtering and downsampling
3. Save subsetted RNA AnnData (cisTopic object is shared — subsetting destroys the topic model)
4. Initialize Snakemake directory
5. Write config.yaml

In [16]:
# ---------------------------------------------------------
# NEW: Global Thresholding Block (Add this before your loop)
# ---------------------------------------------------------

MIN_CELLS_ACROSS_ALL = 50

# 1. Find cell types where ALL species have at least 50 cells
valid_cell_types = comp_df[(comp_df >= MIN_CELLS_ACROSS_ALL).all(axis=1)].index

# 2. See what got dropped so you know what happened
dropped_cts = [ct for ct in comp_df.index if ct not in valid_cell_types]
print(f"\nDropping {len(dropped_cts)} cell types because at least one species had < {MIN_CELLS_ACROSS_ALL} cells:")
print(dropped_cts)

# 3. Filter your target_per_ct to ONLY include the globally valid ones
if target_per_ct is not None:
    target_per_ct = target_per_ct.loc[target_per_ct.index.isin(valid_cell_types)]

print(f"\nProceeding with {len(valid_cell_types)} perfectly harmonized cell types.")

# ---------------------------------------------------------


Dropping 22 cell types because at least one species had < 50 cells:
['Adipocytes', 'BEST4+ cells', 'Enteric neurons', 'Eosinophils', 'Goblet-like cells', 'ICCs', 'ILCs', 'MARCO+ Lymphatic ECs', 'MUC6+ cells', 'Mast cells', 'Mesothelial cells', 'Monocytes', 'Neutrophils', 'Paneth cells', 'Pericytes', 'RBP2_high cells', 'Specialized Fibroblasts ADAMTS16+', 'Specialized Fibroblasts PCDH9+', 'Specialized Fibroblasts RALYL+', 'Specialized Fibroblasts RSPO2/3+', 'Specialized Fibroblasts VCAM1+', 'Tuft cells']

Proceeding with 19 perfectly harmonized cell types.


In [17]:
run_summary = []

for species in SPECIES_LIST:
    print(f"\n{'='*60}")
    print(f"  {species}")
    print(f"{'='*60}")

    species_cfg = SPECIES_CONFIG[species]

    # Load RNA AnnData
    rna_path = os.path.join(INPUT_FILES_DIR, f"{species}_adata_rna_sub.h5ad")
    adata_rna = sc.read_h5ad(rna_path)
    print(f"  Loaded RNA: {adata_rna.shape[0]} cells x {adata_rna.shape[1]} genes")

    # Load cell type info
    meta = cell_meta[species]
    common_idx = adata_rna.obs_names.intersection(meta.index)
    adata_rna_with_ct = adata_rna[common_idx].copy()
    adata_rna_with_ct.obs[CELL_TYPE_COL] = meta.loc[common_idx, CELL_TYPE_COL].values

    # cisTopic object path (shared across seeds)
    cistopic_path = os.path.join(INPUT_FILES_DIR, f"{species}_cistopic_object_subset.pkl")

    for seed in SEEDS:
        print(f"\n  --- Seed {seed} ---")

        selected_cells = downsample_cells(
            meta_df=adata_rna_with_ct.obs,
            cell_type_col=CELL_TYPE_COL,
            target_per_ct=target_per_ct,
            seed=seed,
        )

        adata_sub = adata_rna_with_ct[selected_cells].copy()
        n_cells = adata_sub.shape[0]
        ct_counts = adata_sub.obs[CELL_TYPE_COL].value_counts()
        print(f"  Selected {n_cells} cells ({len(ct_counts)} cell types)")

        run_name = build_run_name(species, seed, ds_label, CELL_TYPES_SUBSET)
        run_dir = os.path.join(SCENICPLUS_DIR, run_name)

        sub_rna_fname = build_sub_rna_name(species, seed, ds_label, CELL_TYPES_SUBSET)
        sub_rna_path = os.path.join(INPUT_FILES_DIR, sub_rna_fname)
        adata_sub.write_h5ad(sub_rna_path)
        print(f"  Saved RNA: {sub_rna_path}")

        initialized = init_snakemake_dir(run_dir)
        if initialized:
            print(f"  Initialized Snakemake: {run_dir}")
        else:
            print(f"  Snakemake dir already exists: {run_dir}")

        config = generate_config_yaml(
            species=species,
            seed=seed,
            cistopic_path=cistopic_path,
            rna_path=sub_rna_path,
            species_cfg=species_cfg,
            temp_dir=TEMP_DIR,
            n_cpu=N_CPU,
        )

        snakemake_dir = os.path.join(run_dir, "Snakemake")
        config_path = os.path.join(snakemake_dir, "config", "config.yaml")
        write_config(config, config_path)
        print(f"  Wrote config: {config_path}")

        run_summary.append({
            "species": species,
            "seed": seed,
            "downsample": ds_label,
            "cell_types": len(ct_counts),
            "n_cells": n_cells,
            "run_dir": run_name,
            "run_dir_abs": run_dir,
            "snakemake_dir": snakemake_dir,
            "config_path": config_path,
            "rna_file": sub_rna_fname,
            "rna_path": sub_rna_path,
        })

    del adata_rna, adata_rna_with_ct

run_df = pd.DataFrame(run_summary)
RUN_MANIFEST_PATH = os.path.join(MANIFESTS_DIR, f"scenicplus_runs_{ds_label}.tsv")
write_run_manifest(run_df, RUN_MANIFEST_PATH)

print("\nDone generating all runs.")
print(f"Run manifest: {RUN_MANIFEST_PATH}")


  Human
  Loaded RNA: 82197 cells x 44020 genes

  --- Seed 42 ---
  Selected 12956 cells (19 cell types)
  Saved RNA: /home/jjanssens/jjans/analysis/adult_intestine/scenicplus/input_files/Human_adata_rna_sub_seed42_dsMin.h5ad
  Snakemake dir already exists: /home/jjanssens/jjans/analysis/adult_intestine/scenicplus/scplus_pipeline_Human_seed42_dsMin
  Wrote config: /home/jjanssens/jjans/analysis/adult_intestine/scenicplus/scplus_pipeline_Human_seed42_dsMin/Snakemake/config/config.yaml

  --- Seed 123 ---
  Selected 12956 cells (19 cell types)
  Saved RNA: /home/jjanssens/jjans/analysis/adult_intestine/scenicplus/input_files/Human_adata_rna_sub_seed123_dsMin.h5ad
  Snakemake dir already exists: /home/jjanssens/jjans/analysis/adult_intestine/scenicplus/scplus_pipeline_Human_seed123_dsMin
  Wrote config: /home/jjanssens/jjans/analysis/adult_intestine/scenicplus/scplus_pipeline_Human_seed123_dsMin/Snakemake/config/config.yaml

  --- Seed 666 ---
  Selected 12956 cells (19 cell types)
  Sa

## Run summary

In [18]:
if "run_df" not in globals():
    run_df = pd.DataFrame(run_summary)

display(run_df)

print(f"\nTotal runs: {len(run_df)}")
print(f"Species: {run_df['species'].nunique()}")
print(f"Seeds: {sorted(run_df['seed'].unique())}")
print(f"Manifest: {RUN_MANIFEST_PATH}")

,species,seed,downsample,cell_types,n_cells,run_dir,run_dir_abs,snakemake_dir,config_path,rna_file,rna_path
0,Human,42,dsMin,19,12956,scplus_pipeline_Human_seed42_dsMin,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,Human_adata_rna_sub_seed42_dsMin.h5ad,/home/jjanssens/jjans/analysis/adult_intestine...
1,Human,123,dsMin,19,12956,scplus_pipeline_Human_seed123_dsMin,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,Human_adata_rna_sub_seed123_dsMin.h5ad,/home/jjanssens/jjans/analysis/adult_intestine...
2,Human,666,dsMin,19,12956,scplus_pipeline_Human_seed666_dsMin,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,Human_adata_rna_sub_seed666_dsMin.h5ad,/home/jjanssens/jjans/analysis/adult_intestine...
3,Gorilla,42,dsMin,19,12956,scplus_pipeline_Gorilla_seed42_dsMin,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,Gorilla_adata_rna_sub_seed42_dsMin.h5ad,/home/jjanssens/jjans/analysis/adult_intestine...
4,Gorilla,123,dsMin,19,12956,scplus_pipeline_Gorilla_seed123_dsMin,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,Gorilla_adata_rna_sub_seed123_dsMin.h5ad,/home/jjanssens/jjans/analysis/adult_intestine...
5,Gorilla,666,dsMin,19,12956,scplus_pipeline_Gorilla_seed666_dsMin,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,Gorilla_adata_rna_sub_seed666_dsMin.h5ad,/home/jjanssens/jjans/analysis/adult_intestine...
6,Chimpanzee,42,dsMin,19,12956,scplus_pipeline_Chimpanzee_seed42_dsMin,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,Chimpanzee_adata_rna_sub_seed42_dsMin.h5ad,/home/jjanssens/jjans/analysis/adult_intestine...
7,Chimpanzee,123,dsMin,19,12956,scplus_pipeline_Chimpanzee_seed123_dsMin,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,Chimpanzee_adata_rna_sub_seed123_dsMin.h5ad,/home/jjanssens/jjans/analysis/adult_intestine...
8,Chimpanzee,666,dsMin,19,12956,scplus_pipeline_Chimpanzee_seed666_dsMin,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,Chimpanzee_adata_rna_sub_seed666_dsMin.h5ad,/home/jjanssens/jjans/analysis/adult_intestine...
9,Bonobo,42,dsMin,19,12956,scplus_pipeline_Bonobo_seed42_dsMin,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,Bonobo_adata_rna_sub_seed42_dsMin.h5ad,/home/jjanssens/jjans/analysis/adult_intestine...



Total runs: 18
Species: 6
Seeds: [42, 123, 666]
Manifest: /home/jjanssens/jjans/analysis/adult_intestine/scenicplus/manifests/scenicplus_runs_dsMin.tsv


## Batch execution script

Generate one executable script for all runs, plus shared post-processing steps that should run only once per species.

In [19]:
PROJECT_ROOT = str(Path(PREP_CONFIG_PATH).resolve().parent.parent)
LINKS_OUT_DIR = os.path.join(SCENICPLUS_DIR, "consensus_links")
LINKS_SCRIPT = os.path.join(PROJECT_ROOT, "scripts", "aggregate_scenic_links.py")

shared_post_commands = {}

link_cfg = SHARED_TASKS.get("link_consensus", {})
if link_cfg.get("enabled", True):
    min_seed_support = int(link_cfg.get("min_seed_support", 2))
    for species in SPECIES_LIST:
        shared_post_commands.setdefault(species, []).append(
            f"python {LINKS_SCRIPT} --scenicplus-dir {SCENICPLUS_DIR} --species {species} --output-dir {LINKS_OUT_DIR} --min-seed-support {min_seed_support}"
        )

motif_cfg = SHARED_TASKS.get("motif_dar_topic", {})
if motif_cfg.get("enabled", False):
    cmd_template = motif_cfg.get("command_template", "").strip()
    if cmd_template:
        for species in SPECIES_LIST:
            shared_post_commands.setdefault(species, []).append(
                cmd_template.format(species=species, scenicplus_dir=SCENICPLUS_DIR, project_root=PROJECT_ROOT)
            )

BATCH_SCRIPT_PATH = os.path.join(SCRIPTS_DIR, f"run_scenicplus_{ds_label}.sh")
generate_batch_script(
    run_df=run_df,
    script_path=BATCH_SCRIPT_PATH,
    n_cpu=N_CPU,
    extra_snakemake_args=SNAKEMAKE_EXTRA_ARGS,
    log_dir=LOGS_DIR,
    shared_post_commands=shared_post_commands,

)

print(f"Batch script written: {BATCH_SCRIPT_PATH}")
print("To run manually:")
print(f"bash {BATCH_SCRIPT_PATH}")

if AUTO_EXECUTE_BATCH:
    print("\nAUTO_EXECUTE_BATCH=True: launching batch script now...")
    subprocess.run(["bash", BATCH_SCRIPT_PATH], check=True)
    print("Batch execution finished.")

Batch script written: /home/jjanssens/jjans/analysis/adult_intestine/scenicplus/scripts/run_scenicplus_dsMin.sh
To run manually:
bash /home/jjanssens/jjans/analysis/adult_intestine/scenicplus/scripts/run_scenicplus_dsMin.sh
